## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-12-02-40-15 +0000,nyt,Nvidia’s Stock Price Plummets as Board Is Said...,https://www.nytimes.com/2026/11/11/business/nv...
1,2026-11-12-02-39-19 +0000,nyt,"Pope Leo XIV Goes Home to Peru, Meeting Ghosts...",https://www.nytimes.com/2026/11/11/world/europ...
2,2026-11-12-02-37-00 +0000,wsj,Dovish BOJ Member Urges Caution in Raising Rat...,https://www.wsj.com/economy/central-banking/do...
3,2026-11-12-02-35-41 +0000,nypost,NYC parents slam construction of men’s homeles...,https://nypost.com/2026/11/11/us-news/nyc-pare...
4,2026-11-12-02-34-54 +0000,nypost,Frontier flight forced to make emergency landi...,https://nypost.com/2026/11/11/us-news/frontier...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2470/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
122,trump,34
0,nvidia,22
91,colom,18
636,war,18
95,mississippi,17


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
17,2026-11-12-02-00-00 +0000,wsj,Democratic Sens. Elizabeth Warren (Mass.) and ...,https://www.wsj.com/politics/policy/top-democr...,168
18,2026-11-12-02-00-00 +0000,wsj,"From the outset, some Trump administration off...",https://www.wsj.com/politics/national-security...,140
112,2026-11-11-22-25-23 +0000,nypost,Nvidia’s Jensen Huang also had alleged affair ...,https://nypost.com/2026/11/11/business/nvidias...,137
155,2026-11-11-21-13-00 +0000,wsj,"Democrats, fresh off sweeping midterm wins, ar...",https://www.wsj.com/politics/national-security...,123
317,2026-11-11-11-53-00 +0000,wsj,House Republicans saved Pete Hegseth and picke...,https://www.wsj.com/politics/policy/house-gop-...,107


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
17,168,2026-11-12-02-00-00 +0000,wsj,Democratic Sens. Elizabeth Warren (Mass.) and ...,https://www.wsj.com/politics/policy/top-democr...
103,96,2026-11-11-22-39-00 +0000,wsj,U.S. Treasury Secretary Scott Bessent was rece...,https://www.wsj.com/politics/national-security...
26,82,2026-11-12-01-30-04 +0000,nyt,Mississippi Democrats built a coalition to ele...,https://www.nytimes.com/live/2026/11/11/us/mis...
317,73,2026-11-11-11-53-00 +0000,wsj,House Republicans saved Pete Hegseth and picke...,https://www.wsj.com/politics/policy/house-gop-...
53,60,2026-11-12-00-24-23 +0000,chitri,"Chicago parents, students protest Google’s new...",https://www.chicagotribune.com/2026/11/11/cps-...
155,57,2026-11-11-21-13-00 +0000,wsj,"Democrats, fresh off sweeping midterm wins, ar...",https://www.wsj.com/politics/national-security...
13,56,2026-11-12-02-02-37 +0000,bbc,Why Democrat Scott Colom’s win in Mississippi ...,https://www.bbc.co.uk/news/videos/cjy5x42p3qzk...
275,53,2026-11-11-14-40-48 +0000,nypost,Joe Biden claims he would have resigned if he ...,https://nypost.com/2026/11/11/us-news/joe-bide...
208,47,2026-11-11-18-00-58 +0000,nypost,California ice cream shop closes after 33-year...,https://nypost.com/2026/11/11/business/family-...
16,43,2026-11-12-02-00-00 +0000,wsj,“The ass-kicking we needed.” South Carolina Re...,https://www.wsj.com/politics/elections/republi...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
